# Tools and tool-execution loops

In [1]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool

# Load GROQ_API_KEY from the local .env file.
load_dotenv()

groq_model = init_chat_model(
    "openai/gpt-oss-120b",
    model_provider="groq",
)


## Define a small, safe tool

In [2]:
@tool  # This decorator registers the function as a tool that can be used in LangChain.
def get_course_module(module_number: int) -> str:
    """Return the topic for a course module number from 1 through 7."""
    modules = {
        1: "AI foundations",
        2: "environment setup and direct model calls",
        3: "LangChain and model settings",
        4: "responses and prompts",
        5: "streaming, batching, structured output, and messages",
        6: "conversation memory and checkpointers",
        7: "tools and tool-execution loops",
    }
    return modules.get(module_number, "That module number is not available.")


course_tools = [
    get_course_module
]  # This list can be expanded with more tools as needed.
tools_by_name = {course_tool.name: course_tool for course_tool in course_tools}


## Let the model request the tool

In [3]:
tool_enabled_model = groq_model.bind_tools(course_tools)

messages = [
    SystemMessage("Use the course tool when the learner asks about a module number."),
    HumanMessage("What will I learn in module 6?"),
]

model_message = tool_enabled_model.invoke(messages)
print("Requested tool calls:", model_message.tool_calls)


Requested tool calls: [{'name': 'get_course_module', 'args': {'module_number': 6}, 'id': 'fc_c74e998f-b543-4fed-8d9b-274c89b8aedc', 'type': 'tool_call'}]


## Execute requested tools and return the result

In [4]:
messages.append(model_message)

for tool_call in model_message.tool_calls:
    requested_tool = tools_by_name.get(tool_call["name"])
    if requested_tool is None:
        # Do not execute a tool that the application did not explicitly allow.
        tool_result = "This tool is not available."
    else:
        tool_result = requested_tool.invoke(tool_call["args"])

    messages.append(
        ToolMessage(
            content=str(tool_result),
            tool_call_id=tool_call["id"],
        )
    )

final_response = tool_enabled_model.invoke(messages)
print(final_response.text)


In **Module 6** you’ll dive into **Conversation Memory and Checkpointers**. This module covers how to store, retrieve, and manage contextual information across interactions, as well as techniques for creating checkpoints that let you resume or branch conversations reliably. You’ll learn how to:

- Structure memory so the system can recall relevant details later.
- Implement checkpointers to save and restore conversation states.
- Use memory effectively to maintain coherent, personalized dialogues.
- Handle edge cases like memory limits, privacy concerns, and context drift.

By the end of the module, you’ll be able to build applications that remember past interactions and seamlessly continue conversations from any saved point.
